## Homework assignment 2: Recurrent neural networks

### Assessment and penalties

The maximum grade allowed for this assignment is __10 (+3)__ points. Submitting an assignment after the hard deadline is prohibited. Submitting a solution after the soft deadline will deduct __one__ point for each day late.

The assignment must be completed independently. "Similar" solutions are considered plagiarism, and all students involved (including those from whom the plagiarism was committed) cannot receive more than 0 points for it. All code must be written independently. Using someone else's code is prohibited, even with a link to the source. Within reasonable limits, of course. Taking a couple of obvious lines of code to implement a small feature is acceptable.

Ineffective code implementation may negatively impact your grade. Your grade may also be reduced for poorly readable code.

__Soft deadline: 09.10.25 23:59__   
__Hard deadline: 11.10.25 23:59__

### About task

In this task, you will implement an LSTM model to solve a multi-label classification problem. This is a type of classification in which each object can belong to multiple classes simultaneously. This problem often arises when classifying films by genre, scientific or news articles by topic, musical compositions by instrument, and so on.

In our case, we will work with a dataset of biotech news and classify them by topic. This dataset is already preprocessed: the text is converted to lowercase, punctuation is removed, and all words are separated by spaces.

In [ ]:
import pandas as pd

dataset = pd.read_csv('biotech_news.tsv', sep='\t')
dataset.head()

## Label Preprocessing

__Task 1 (0.5 points)__. As you can see, the labels are written as comma-separated strings. To work with them, we need to convert them to numbers. Since each object can belong to multiple classes, encode the labels as vectors of 0 and 1, where 1 means the object belongs to the corresponding class, and 0 means it does not. With this encoding, we can train the model by solving a binary classification problem for each class.

In [ ]:
# your code here

## Data preprocessing

In this task, we'll train recurrent neural networks. As you know, they work better with short texts because they're not very good at capturing long-range relationships. To reduce text length, it should be cleaned.

We'll immediately split the dataset into training and test datasets so that all the necessary statistics are calculated using only the training dataset.

In [ ]:
from sklearn.model_selection import train_test_split

texts_train, texts_test, y_train, y_test = train_test_split(
    <texts>,
    <processed labels>,
    test_size=0.2,  # do not change this
    random_state=0  # do not change this
)

__Task 2 (1 point)__. Remove stop words, overly rare words, and overly frequent words from the texts. Choose the hyperparameters yourself (ideally, they should be selected based on the quality on the test sample). If you think you need to add any additional processing, do so. It's important not to remove anything that could affect the class prediction.

In [ ]:
# your code here

__Task 3 (1.5 points)__. The remaining step is to convert the texts into token indexes so they can be fed into the model. You have two options for how to do this:
1. __(+0 points)__ Tokenize the texts by words.
2. __(up to +3 points)__ Implement your own BPE tokenization. The number of points will vary depending on the effectiveness of the implementation. You cannot use specialized libraries during the implementation.

Tokenize the texts, convert them into index lists, and combine them with labels in the DataLoader. Don't forget to add `collate_fn` to the DataLoader, which will pad all short texts in the batch. You may find `gensim.corpora.dictionary.Dictionary` useful for mapping tokens to indexes.

In [ ]:
# your code here

## Quality Metric

Before training, we need to choose a quality evaluation metric. Since classes in a multi-label classification are often imbalanced, the [F1 score](https://en.wikipedia.org/wiki/F-score) is most often used as the metric.

The `compute_f1` function takes the true and predicted labels and calculates the average F1 score across all classes. Use it to evaluate the quality of models.

$$
F1_{total} = \frac{1}{K} \sum_{k=1}^K F1(Y_k, \hat{Y}_k),
$$
where $Y_k$ are the true values ​​for class k, and $\hat{Y}_k$ are the predictions.

In [ ]:
from sklearn.metrics import f1_score

def compute_f1(y_true, y_pred):
    assert y_true.ndim == 2
    assert y_true.shape == y_pred.shape

    return f1_score(y_true, y_pred, average='macro')

## Model training

### RNN

For the baseline, we'll train the simplest recurrent neural network. As a reminder, an RNN block looks like this.

<img src="https://i.postimg.cc/yYbNBm6G/tg-image-1635618906.png" alt="drawing" width="400"/>

Its hidden state is updated as
$h_t = \sigma(W x_{t} + U h_{t-1} + b_h)$. The prediction is calculated by applying a linear layer to the last token
$o_T = V h_T + b_o$. Choose the hyperbolic tangent as the activation function.

__Task 4 (2 points)__. Implement an RNN according to the formula above and train it on the provided dataset. Initialize the hidden state vector with zeros; this will ensure more stable learning than with random initialization. Afterwards, measure the performance on the test set. You should achieve an F1 value of at least 0.33, and the training itself shouldn't take long.

In [ ]:
# your code here

### LSTM

<img src="https://i.postimg.cc/pL5LdmpL/tg-image-2290675322.png" alt="drawing" width="400"/>

Now let's move on to more advanced recurrent models, namely LSTM. Due to the additional memory vector, this model should be much better at capturing distant dependencies, which should directly impact performance.

The LSTM block parameters are updated as follows ($\sigma$ stands for sigmoid):
\begin{align}
f_{t} &= \sigma(W_f x_{t} + U_f h_{t-1} + b_f) \\ 
i_{t} &= \sigma(W_i x_{t} + U_i h_{t-1} + b_i) \\
\tilde{c}_{t} &= \tanh(W_c x_{t} + U_c h_{t-1} + b_i) \\
c_{t} &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t \\
o_{t} &= \sigma(W_t x_{t} + U_t h_{t-1} + b_t) \\
h_t &= o_t \odot \tanh(c_t)
\end{align}

__Task 5 (2 points).__ Implement an LSTM using the described scheme. Choose the LSTM hyperparameters so that their total number (excluding the embedding layer) is approximately the same as the number of parameters of a regular RNN, but the hidden layer dimension is at least 64. This way, we can compare the architectures as independently as possible. Train the LSTM to convergence and compare its performance with the RNN on the test set. Did you achieve a better result? How can you explain this?

In [ ]:
# your code here

__Task 6 (2 points).__ The main drawback of RNN models is that when compressing all the information into a single vector, important details are lost. To solve this problem, an attention mechanism was invented. Implement it according to the [original paper](https://arxiv.org/abs/1409.0473). Measure the quality and draw conclusions.

Note that this method was proposed for Encoder-Decoder models. In our case, there is no decoder, so embed attention in the encoder: each LSTM block will look at the outputs of all previous blocks.

In [ ]:
# your code here

__Task 7 (1 point).__ Add the ability to increase the number of LSTM layers to your implementation. Train a model with two layers and measure the performance. Draw your own conclusions: is it worth increasing the model size?

In [ ]:
# your code here